In [3]:
from datetime import datetime, timedelta
from ordered_set import OrderedSet
import numpy as np
from scipy.stats import uniform

from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, ConstantVelocity
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.types.detection import TrueDetection, Clutter
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.types.state import GaussianState
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.updater.kalman import KalmanUpdater
from stonesoup.deleter.time import UpdateTimeDeleter
from stonesoup.initiator.simple import MultiMeasurementInitiator
from stonesoup.hypothesiser.probability import PDAHypothesiser
from stonesoup.dataassociator.probability import JPDA
from stonesoup.tracker.simple import MultiTargetTracker
from stonesoup.types.multihypothesis import MultipleHypothesis
from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
from stonesoup.measures import Euclidean
from stonesoup.metricgenerator.manager import MultiManager
from stonesoup.dataassociator.tracktotrack import TrackToTruth

import matplotlib.pyplot as plt
import scienceplots

# =============================================================================
# 1. Generate ground truth for two objects
# =============================================================================
np.random.seed(1991)
truths = OrderedSet()
num_steps = 20
start_time = datetime.now().replace(microsecond=0)
transition_model = CombinedLinearGaussianTransitionModel([
    ConstantVelocity(0.005),
    ConstantVelocity(0.005)
])

timesteps = [start_time]
# first target
truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=start_time)])
for k in range(1, num_steps + 1):
    t = start_time + timedelta(seconds=k)
    timesteps.append(t)
    next_state = transition_model.function(
        truth[k-1],
        noise=True,
        time_interval=timedelta(seconds=1)
    )
    truth.append(GroundTruthState(next_state, timestamp=t))
truths.add(truth)

# second target
truth = GroundTruthPath([GroundTruthState([0, 1, 20, -1], timestamp=start_time)])
for k in range(1, num_steps + 1):
    t = start_time + timedelta(seconds=k)
    next_state = transition_model.function(
        truth[k-1],
        noise=True,
        time_interval=timedelta(seconds=1)
    )
    truth.append(GroundTruthState(next_state, timestamp=t))
truths.add(truth)

# =============================================================================
# 2. Generate measurements (with clutter)
# =============================================================================
measurement_model = LinearGaussian(
    ndim_state=4,
    mapping=(0, 2),
    noise_covar=np.array([[0.75, 0], [0, 0.75]])
)
prob_detect = 0.80
all_measurements = []

for k in range(num_steps):
    measurement_set = set()
    for truth in truths:
        # missed detection
        if np.random.rand() <= prob_detect:
            meas = measurement_model.function(truth[k], noise=True)
            measurement_set.add(
                TrueDetection(
                    state_vector=meas,
                    groundtruth_path=truth,
                    timestamp=truth[k].timestamp,
                    measurement_model=measurement_model
                )
            )
        # clutter
        for _ in range(np.random.randint(2, 5)):
            x = uniform.rvs(truth[k].state_vector[0] - 10, 20)
            y = uniform.rvs(truth[k].state_vector[2] - 10, 20)
            measurement_set.add(
                Clutter(
                    np.array([[x], [y]]),
                    timestamp=truth[k].timestamp,
                    measurement_model=measurement_model
                )
            )
    all_measurements.append(measurement_set)

# =============================================================================
# 3. Build JPDAF‐based MultiTargetTracker
# =============================================================================
# common pieces
predictor = KalmanPredictor(transition_model)
updater   = KalmanUpdater(measurement_model)
deleter   = UpdateTimeDeleter(timedelta(seconds=4), delete_last_pred=True)

# a tiny subclass to pick the best hypothesis out of a MultipleHypothesis
class JPDAFInitiator(MultiMeasurementInitiator):
    def initiate(self, detections, timestamp, **kwargs):
        new = super().initiate(detections, timestamp, **kwargs)
        # Only resolve the MultipleHypothesis in newly created tracks
        for track in new:
            if isinstance(track[-1], MultipleHypothesis):
                best = max(track[-1].hypotheses, key=lambda h: h.weight)
                track[-1] = best
        return new


hypothesiser = PDAHypothesiser(
    predictor,
    updater,
    clutter_spatial_density=0.25,
    prob_detect=prob_detect,
    prob_gate=0.999
)
jpda_assoc = JPDA(hypothesiser)

initiator = JPDAFInitiator(
    prior_state     = GaussianState([[0], [0], [0], [0]], np.diag([100,10,100,10])),
    measurement_model=measurement_model,
    deleter         = deleter,
    data_associator = jpda_assoc,
    updater         = updater,
    min_points      = 5
)

# make the detector generator
detector = ((timesteps[k], all_measurements[k]) for k in range(num_steps))

tracker = MultiTargetTracker(
    initiator       = initiator,
    deleter         = deleter,
    detector        = detector,
    data_associator = jpda_assoc,
    updater         = updater
)
tracker.predictor = predictor


# =============================================================================
# 4. Run tracker, collect tracks of length ≥5
# =============================================================================
all_tracks = set()
for time, curr in tracker:
    all_tracks |= curr

tracks = {t for t in all_tracks if len(t.states) >= 5}

# =============================================================================
# 5. Compute SIAP metrics
# =============================================================================
associator = TrackToTruth(association_threshold=30)
siap_metrics = SIAPMetrics(
    position_measure = Euclidean((0,2)),
    velocity_measure = Euclidean((1,3)),
    generator_name   = 'SIAP',
    tracks_key       = 'tracks',
    truths_key       = 'truths'
)
mm = MultiManager([siap_metrics], associator=associator)
mm.add_data({'tracks': tracks}, overwrite=False)

flat_dets = set().union(*all_measurements)
mm.add_data({'truths': truths, 'detections': flat_dets}, overwrite=False)

siap = mm.generate_metrics()['SIAP']

# unwrap
s_avg  = siap.get('SIAP Spuriousness').value
a_avg  = siap.get('SIAP Ambiguity').value
p_avg  = siap.get('SIAP Position Accuracy').value
v_avg  = siap.get('SIAP Velocity Accuracy').value

s_time = [m.value for m in siap.get('SIAP Spuriousness at times').value]
a_time = [m.value for m in siap.get('SIAP Ambiguity at times').value]
p_time = [m.value for m in siap.get('SIAP Position Accuracy at times').value]
v_time = [m.value for m in siap.get('SIAP Velocity Accuracy at times').value]

times = timesteps[:1+len(s_time)]

# =============================================================================
# 6. Plot results
# =============================================================================
plt.style.use(['science','no-latex'])
fig, axs = plt.subplots(2,2,figsize=(12,9))
fig.suptitle("SIAP Metrics Over Time")

# spuriousness
axs[0,0].plot(times, s_time, 'o-', label='Spuriousness'); axs[0,0].set_ylim(0,1)
# ambiguity
axs[0,1].plot(times, a_time, 'o-', label='Ambiguity'); axs[0,1].set_ylim(0,2)
# position
axs[1,0].plot(times, p_time, 'o-', label='PosErr')
# velocity
axs[1,1].plot(times, v_time, 'o-', label='VelErr')

for ax in axs.flat:
    ax.legend(); ax.grid(True)

plt.tight_layout(); plt.show()

# bar chart of averages
fig, ax = plt.subplots(1,4,figsize=(14,4))
ax[0].bar("Spur", s_avg); ax[1].bar("Amb", a_avg)
ax[2].bar("PosErr", p_avg); ax[3].bar("VelErr", v_avg)
for a, val in zip(ax, (s_avg,a_avg,p_avg,v_avg)):
    a.set_ylim(0, max(1.5,val*1.2))
    a.text(0,val*1.05,f"{val:.2f}",ha='center')
plt.tight_layout(); plt.show()


AttributeError: 'MultipleHypothesis' object has no attribute 'prediction'

In [ ]:
# Extra Stuff
"""
from datetime import datetime, timedelta
from ordered_set import OrderedSet
import numpy as np
from scipy.stats import uniform
import matplotlib.pyplot as plt

# --- Stone Soup imports ---
from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, ConstantVelocity
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.types.detection import TrueDetection, Clutter
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.types.state import GaussianState
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.updater.kalman import KalmanUpdater
from stonesoup.deleter.time import UpdateTimeDeleter
from stonesoup.initiator.simple import MultiMeasurementInitiator
from stonesoup.dataassociator.neighbour import NearestNeighbour
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.measures import Mahalanobis
from stonesoup.tracker.simple import MultiTargetTracker
from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
from stonesoup.measures import Euclidean
from stonesoup.metricgenerator.manager import MultiManager
from stonesoup.dataassociator.tracktotrack import TrackToTruth

# -----------------------------------------
# 1) GENERATE GROUND TRUTH
# -----------------------------------------
np.random.seed(1991)
truths = OrderedSet()
num_steps  = 20
start_time = datetime.now().replace(microsecond=0)
transition_model = CombinedLinearGaussianTransitionModel([
    ConstantVelocity(0.005),
    ConstantVelocity(0.005)
])

timesteps = [start_time]
# Truth #1
truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=start_time)])
for k in range(1, num_steps+1):
    t = start_time + timedelta(seconds=k)
    timesteps.append(t)
    truth.append(
        GroundTruthState(
            transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=1)),
            timestamp=t
        )
    )
truths.add(truth)

# Truth #2
truth = GroundTruthPath([GroundTruthState([0, 1, 20, -1], timestamp=start_time)])
for k in range(1, num_steps+1):
    t = start_time + timedelta(seconds=k)
    truth.append(
        GroundTruthState(
            transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=1)),
            timestamp=t
        )
    )
truths.add(truth)


# -----------------------------------------
# 2) GENERATE MEASUREMENTS (Detections + Clutter)
# -----------------------------------------
measurement_model = LinearGaussian(
    ndim_state=4, mapping=(0,2),
    noise_covar=np.array([[0.75, 0],[0,0.75]])
)
prob_detect = 0.80
all_measurements = []

for k in range(num_steps):
    ms = set()
    for truth in truths:
        # True detection?
        if np.random.rand() <= prob_detect:
            meas = measurement_model.function(truth[k], noise=True)
            ms.add(
                TrueDetection(
                    state_vector=meas,
                    groundtruth_path=truth,
                    timestamp=truth[k].timestamp,
                    measurement_model=measurement_model
                )
            )
        # Clutter
        for _ in range(np.random.randint(5,12)):
            x = uniform.rvs(truth[k].state_vector[0]-10, 20)
            y = uniform.rvs(truth[k].state_vector[2]-10, 20)
            ms.add(
                Clutter(
                    np.array([[x],[y]]),
                    timestamp=truth[k].timestamp,
                    measurement_model=measurement_model
                )
            )
    all_measurements.append(ms)


# -----------------------------------------
# 3) TRACKER SETUP (shared)
# -----------------------------------------
predictor      = KalmanPredictor(transition_model)
updater        = KalmanUpdater(measurement_model)
deleter        = UpdateTimeDeleter(timedelta(seconds=4), delete_last_pred=True)
hypothesiser   = DistanceHypothesiser(predictor, updater, Mahalanobis(), missed_distance=4)
data_associator= NearestNeighbour(hypothesiser)
prior_state    = GaussianState([[0],[1],[0],[1]], np.diag([1,1,1,1]))
initiator      = MultiMeasurementInitiator(
    prior_state=prior_state,
    measurement_model=measurement_model,
    deleter=deleter,
    data_associator=data_associator,
    updater=updater,
    min_points=3
)


# -----------------------------------------
# 4) RUN TRACKER #1: Classic Kalman
# -----------------------------------------
detector1 = ((timesteps[k], all_measurements[k]) for k in range(num_steps))
tracker1  = MultiTargetTracker(
    initiator=initiator,
    deleter=deleter,
    detector=detector1,
    data_associator=data_associator,
    updater=updater
)
tracks1 = set()
for t, current in tracker1:
    tracks1 |= current


# -----------------------------------------
# 5) RUN TRACKER #2: "Influence Diagram" Kalman
# (same config for demo; substitute your custom updater/predictor here)
# -----------------------------------------
detector2 = ((timesteps[k], all_measurements[k]) for k in range(num_steps))
tracker2  = MultiTargetTracker(
    initiator=initiator,
    deleter=deleter,
    detector=detector2,
    data_associator=data_associator,
    updater=updater   # <— swap with your ID‑Kalman updater!
)
tracks2 = set()
for t, current in tracker2:
    tracks2 |= current


# -----------------------------------------
# 6) PICK THE TWO BEST TRACKS PER RUN
# -----------------------------------------
def best_tracks(all_tracks, truths, n=2):
    chosen = []
    for truth in truths:
        best, best_score = None, float('inf')
        for track in all_tracks:
            # overlap on timestamps
            overlap = [s for s in track.states if s.timestamp in {st.timestamp for st in truth.states}]
            if not overlap: continue
            # mean 2D error
            errs = []
            for s in overlap:
                idx = next(i for i,t in enumerate(truth.states) if t.timestamp==s.timestamp)
                errs.append(np.linalg.norm(
                    s.state_vector[[0,2]] - truth.states[idx].state_vector[[0,2]]
                ))
            score = np.mean(errs)
            if score < best_score:
                best, best_score = track, score
        if best is not None:
            chosen.append(best)
    return chosen

tracks1_best = best_tracks(tracks1, truths)
tracks2_best = best_tracks(tracks2, truths)


# -----------------------------------------
# 7) COMPUTE SIAP METRICS
# -----------------------------------------
def compute_siap(tracks):
    associator = TrackToTruth(association_threshold=30)
    mgen = SIAPMetrics(
        position_measure=Euclidean((0,2)),
        velocity_measure=Euclidean((1,3)),
        generator_name='SIAP', tracks_key='tracks', truths_key='truths'
    )
    mgr = MultiManager([mgen], associator=associator)
    mgr.add_data({'tracks': set(tracks)}, overwrite=False)
    # need truths + detections (flat)
    flat = set.union(*all_measurements)
    mgr.add_data({'truths': truths, 'detections': flat}, overwrite=False)
    return mgr.generate_metrics()['SIAP']

siap1 = compute_siap(tracks1_best)
siap2 = compute_siap(tracks2_best)


# -----------------------------------------
# 8) UNPACK THE NUMERIC LISTS
# -----------------------------------------
def unpack(s):
    out = {}
    out['spurious_avg']    = s.get('SIAP Spuriousness').value
    out['ambiguity_avg']   = s.get('SIAP Ambiguity').value
    out['posacc_avg']      = s.get('SIAP Position Accuracy').value
    out['velacc_avg']      = s.get('SIAP Velocity Accuracy').value
    out['spurious_time']   = [m.value for m in s.get('SIAP Spuriousness at times').value]
    out['ambiguity_time']  = [m.value for m in s.get('SIAP Ambiguity at times').value]
    out['posacc_time']     = [m.value for m in s.get('SIAP Position Accuracy at times').value]
    out['velacc_time']     = [m.value for m in s.get('SIAP Velocity Accuracy at times').value]
    return out

m1 = unpack(siap1)
m2 = unpack(siap2)

# Align time axis
t_plot = timesteps[1:1+len(m1['spurious_time'])]
"""

In [ ]:
# --- after unpacking m1, m2 ---
"""
# Retrieve the raw SingleTimeMetric lists so we can grab their timestamps:
series_defs = [
    ('spurious_time', 'SIAP Spuriousness at times',    'Spuriousness'),
    ('ambiguity_time','SIAP Ambiguity at times',       'Ambiguity'),
    ('posacc_time',   'SIAP Position Accuracy at times','Position Error'),
    ('velacc_time',   'SIAP Velocity Accuracy at times','Velocity Error'),
]

# 1) Time‑series comparison
fig, axs = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("SIAP Metrics Over Time\nClassic vs ID‑Diagram Kalman", fontsize=16)

for ax, (key, metric_name, title) in zip(axs.flat, series_defs):
    # grab the underlying metrics
    series1 = siap1.get(metric_name).value  # list of SingleTimeMetric
    series2 = siap2.get(metric_name).value

    # extract x/y
    t1 = [m.timestamp for m in series1]
    y1 = [m.value     for m in series1]
    t2 = [m.timestamp for m in series2]
    y2 = [m.value     for m in series2]

    ax.plot(t1, y1, 'o-', color=colors[0], label=labels[0])
    ax.plot(t2, y2, 's--', color=colors[1], label=labels[1])
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel(title)
    ax.grid(True)
    ax.legend()

plt.tight_layout(rect=[0,0,1,0.93])
plt.show()


# 2) Bar‑chart of averages (unchanged)
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Average SIAP Metric Comparison", fontsize=14)

avg_defs = [
    ('spurious_avg', 'SIAP Spuriousness',      'Spuriousness'),
    ('ambiguity_avg','SIAP Ambiguity',         'Ambiguity'),
    ('posacc_avg',   'SIAP Position Accuracy', 'Position Error'),
    ('velacc_avg',   'SIAP Velocity Accuracy', 'Velocity Error'),
]

for ax, (key, metric_name, title) in zip(axs, avg_defs):
    v1 = m1[key]
    v2 = m2[key]
    ax.bar(labels, [v1, v2], color=colors)
    ax.set_title(title)
    top = max(v1, v2, 1e-3)
    ax.set_ylim(0, top * 1.2)
    for i, v in enumerate([v1, v2]):
        ax.text(i, v + top * 0.02, f"{v:.2f}", ha='center')

plt.tight_layout(rect=[0,0,1,0.88])
plt.show()
"""